<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/4_Aprendizaje_no_supervisado/2_Taller_Apriori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Análisis de Patrones de Consumo Internacional con Apriori**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Camilo Rendon C.
- Nicolás Cubillos Riveros.
-

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma:“Taller_Apriori_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/qERdEpXpmx.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

21 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

**Caso de Estudio: Consultoría para Global Retail Inc.**

**Contexto:** Una firma multinacional de e-commerce, "Global Retail Inc.", te ha contratado como consultor de datos. La empresa opera en múltiples países y ha notado que sus ventas y la efectividad de sus campañas de marketing varían significativamente entre regiones. Su hipótesis es que los patrones de compra y las asociaciones de productos son diferentes en cada mercado.

**Tu Misión:** Analizar el historial de transacciones de la empresa para descubrir y comparar las reglas de asociación de productos para dos de sus mercados más importantes en Latinoamérica: México y Colombia. Tu objetivo final es entregar recomendaciones de negocio accionables (ej. estrategias de cross-selling, promociones personalizadas) basadas en los patrones de consumo que descubras en cada país.

**Dataset:** Encuentra mayor información en el archivo "diccionario_alimentos_retail_top30.xlsx".

## Ejercicio 1: Configuración Inicial, Carga y Exploración de Datos

1.1 Importa las librerías necesarias

In [1]:
import os
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Configuraciones de visualización
pd.options.display.max_columns = None
pd.options.display.float_format = '{:,.2f}'.format

1.2 Carga el dataset "alimentos_retail_top30.csv" que se encuentra en el repositorio del curso, carpeta "datasets". El dataframe debe llamarse "df".

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path = '/content/drive/MyDrive/Bases'
# Para establecer el directorio de los archivos
os.chdir(path)

In [5]:
### TU CÓDIGO AQUÍ ###
df = pd.read_csv('alimentos_retail_top30.csv')

In [6]:
# Debe ser (6899, 8)
print("Dimensiones del DataFrame:")
print(df.shape)

Dimensiones del DataFrame:
(6899, 8)


In [7]:
print("\nInformación general del DataFrame:")
df.info()


Información general del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6899 entries, 0 to 6898
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    6899 non-null   object 
 1   StockCode    6899 non-null   int64  
 2   Description  6899 non-null   object 
 3   Quantity     6899 non-null   int64  
 4   InvoiceDate  6899 non-null   object 
 5   UnitPrice    6899 non-null   float64
 6   CustomerID   6879 non-null   float64
 7   Country      6899 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 431.3+ KB


1.3 Revisa si hay valores nulos en alguna columna y cuántos son

In [8]:
null = df.isnull().sum()
print(null)

InvoiceNo       0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
UnitPrice       0
CustomerID     20
Country         0
dtype: int64


1.4 Genera las estadísticas descriptivas de las variables numéricas

In [9]:
df.describe()

,StockCode,Quantity,UnitPrice,CustomerID
count,"6,899.00","6,899.00","6,899.00","6,879.00"
mean,"55,544.94",3.00,3.42,"15,024.12"
std,"25,875.73",1.43,1.06,"1,732.95"
min,"26,907.00",-5.00,1.65,"12,000.00"
25%,"31,048.00",2.00,2.36,"13,524.00"
50%,"42,889.00",3.00,3.39,"15,041.00"
75%,"87,297.00",4.00,4.44,"16,530.50"
max,"95,931.00",5.00,4.90,"17,999.00"


1.5 Observando las salidas del ejercicio anterior, ¿qué problemas potenciales identificas en las columnas CustomerID y Quantity?

## Ejercicio 2: Limpieza y Preprocesamiento de Datos

Los datos del mundo real rara vez son perfectos. Antes de cualquier análisis, debemos "sanear" nuestro dataset. Completa el código en cada paso según las instrucciones.

Crea un nuevo dataframe llamado "df_limpio" para los siguientes puntos.

2.1 **Manejo de Valores Nulos**: Las transacciones sin un CustomerID no son útiles para nosotros, ya que no podemos agrupar las compras de un cliente específico.

In [10]:
# TAREA: Elimina todas las filas donde 'CustomerID' es nulo.
### TU CÓDIGO AQUÍ ###
df_limpio = df.dropna(subset=['CustomerID'])

In [11]:
# El tipo de dato de CustomerID debe ser entero
### TU CÓDIGO AQUÍ ###
df_limpio['CustomerID'] = df_limpio['CustomerID'].astype(int)

2.2 **Limpieza de Descripciones de Productos** Las descripciones pueden tener espacios en blanco al inicio o al final que podrían hacer que un mismo producto se cuente como dos diferentes.

In [12]:
# TAREA: # Verifica cuántas descripciones únicas hay.
df_limpio['Description'].nunique()

25

In [13]:
# TAREA: Limpia la columna 'Description' eliminando espacios extra al inicio y al final.
df_limpio['Description'] = df_limpio['Description'].str.strip()

In [14]:
# TAREA: Verifica cuántas descripciones únicas quedaron después de la limpieza.
df_limpio['Description'].nunique()

20

2.3 **Filtrado de Transacciones Anómalas**: Las facturas (InvoiceNo) que empiezan con 'C' indican una cancelación. Estas no son compras reales y deben ser eliminadas. Del mismo modo, las cantidades (Quantity) negativas representan devoluciones.

In [15]:
# TAREA: Elimina las filas que correspondan a cancelaciones.
df_limpio = df_limpio[~df_limpio['InvoiceNo'].str.startswith('C')]

In [16]:
# TAREA: Elimina las filas con cantidades negativas.
df_limpio = df_limpio[df_limpio['Quantity'] >= 0]


In [17]:
# Verifiquemos las dimensiones del DataFrame después de la limpieza. Debe ser (6864, 8)
df_limpio.shape

(6864, 8)

In [ ]:
df_limpio

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536000,94537,HARINA DE MAÍZ,5,2023-01-07 01:09:00,2.76,17452,Colombia
1,536000,87297,QUESO MUZZARELLA,2,2023-01-07 01:09:00,4.69,17779,Colombia
2,536001,94537,HARINA DE MAÍZ,4,2023-01-07 11:51:00,2.76,14933,Colombia
3,536001,87297,QUESO MUZZARELLA,3,2023-01-07 11:51:00,4.69,14957,Colombia
4,536002,26907,CAFÉ,4,2023-01-02 01:54:00,2.36,15202,Colombia
...,...,...,...,...,...,...,...,...
6893,537998,48011,FRIJOL NEGRO,4,2023-01-05 14:28:00,1.86,12401,México
6894,537998,36301,TORTILLAS DE MAÍZ,4,2023-01-05 14:28:00,4.14,13520,México
6895,537999,48011,FRIJOL NEGRO,1,2023-01-07 20:26:00,1.86,12105,México
6897,537999,36301,TORTILLAS DE MAÍZ,2,2023-01-07 20:26:00,4.14,15425,México


## Ejercicio 3: Análisis Comparativo por País

Ahora que los datos están limpios, vamos a segmentarlos y a aplicar el algoritmo Apriori para encontrar los patrones de compra en México y Colombia.

**Preparación de la Cesta de Mercado (Función)**

La siguiente función toma un dataframe, lo agrupa por factura y descripción, y lo transforma en el formato de matriz binaria que necesita el algoritmo Apriori. Estudia esta función, no necesitas modificarla.

In [18]:
def preparar_cesta(dataframe, pais):
    """Filtra por país y prepara la matriz de transacciones."""

    # Filtrar por el país de interés
    df_pais = dataframe[dataframe['Country'] == pais]

    # Crear la cesta: agrupar productos por factura
    cesta = (df_pais.groupby(['InvoiceNo', 'Description'])['Quantity']
             .sum().unstack().reset_index().fillna(0)
             .set_index('InvoiceNo'))

    # Convertir todas las cantidades positivas a 1 y todo lo demás a 0
    cesta_encoded = (cesta > 0).astype(int)

    return cesta_encoded

3.1 Análisis para México

In [19]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de México. Almacena el resultado en la variable cesta_mx.
### TU CÓDIGO AQUÍ ###
cesta_mx = preparar_cesta(df_limpio, 'México')
cesta_mx.head()

Description,AGUACATE,CEBOLLA,CHILE JALAPEÑO,CILANTRO,FRIJOL NEGRO,LIMÓN,QUESO FRESCO,TOMATE,TORTILLAS DE MAÍZ,TOTOPOS
InvoiceNo,,,,,,,,,,
537000,0,0,0,0,1,0,0,0,1,0
537001,0,1,1,1,0,0,0,1,0,0
537002,0,0,0,0,1,0,0,1,1,0
537003,0,1,1,1,0,0,0,1,0,0
537004,0,0,0,1,0,0,1,1,1,0


In [20]:
# TAREA: Aplica el algoritmo apriori para encontrar itemsets con un soporte mínimo de 2%.
# Almacena el resultado en la variable frequent_itemsets_mx.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_mx = apriori(cesta_mx, min_support=0.02, use_colnames=True)
frequent_itemsets_mx.sort_values(by = 'support', ascending=False).head(10)

,support,itemsets
2,0.42,(CHILE JALAPEÑO)
7,0.41,(TOMATE)
3,0.41,(CILANTRO)
1,0.41,(CEBOLLA)
8,0.38,(TORTILLAS DE MAÍZ)
4,0.36,(FRIJOL NEGRO)
0,0.35,(AGUACATE)
5,0.35,(LIMÓN)
9,0.33,(TOTOPOS)
31,0.33,"(TOMATE, CHILE JALAPEÑO)"


In [21]:
# TAREA: Genera las reglas de asociación. Queremos reglas con un Lift mayor a 2. Almacena el resultado en la variable rules_mx.
### TU CÓDIGO AQUÍ ###
rules_mx = association_rules(frequent_itemsets_mx, metric='lift', min_threshold=2)

In [22]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
rules_mx.sort_values(by=['lift', 'confidence'], ascending=[False, False]).head(10)[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]


,antecedents,consequents,antecedent support,consequent support,confidence,lift
93,"(CEBOLLA, CHILE JALAPEÑO)","(CILANTRO, TOMATE)",0.32,0.32,0.92,2.90
88,"(CILANTRO, TOMATE)","(CEBOLLA, CHILE JALAPEÑO)",0.32,0.32,0.93,2.90
91,"(CILANTRO, CEBOLLA)","(TOMATE, CHILE JALAPEÑO)",0.31,0.33,0.95,2.88
90,"(TOMATE, CHILE JALAPEÑO)","(CILANTRO, CEBOLLA)",0.33,0.31,0.90,2.88
11,"(AGUACATE, LIMÓN)",(TOTOPOS),0.27,0.33,0.93,2.82
14,(TOTOPOS),"(AGUACATE, LIMÓN)",0.33,0.27,0.75,2.82
92,"(CILANTRO, CHILE JALAPEÑO)","(TOMATE, CEBOLLA)",0.32,0.33,0.92,2.81
89,"(TOMATE, CEBOLLA)","(CILANTRO, CHILE JALAPEÑO)",0.33,0.32,0.91,2.81
73,"(AGUACATE, TOMATE, LIMÓN)",(TOTOPOS),0.02,0.33,0.91,2.76
76,(TOTOPOS),"(AGUACATE, TOMATE, LIMÓN)",0.33,0.02,0.06,2.76


3.3 Observa las 3 reglas con el Lift más alto para México (1, 3 y 5). **Interprétalas:** ¿Qué te dicen estas asociaciones? ¿Qué tipo de productos son?

Las reglas 1 y 3 muestran que ingredientes como cebolla, chile jalapeño, cilantro y tomate suelen comprarse juntos con frecuencia. Esto refleja un patrón cultural claro, ya que estos productos son la base de preparaciones tradicionales como el pico de gallo, lo que sugiere que los clientes los adquieren pensando en cocinar este tipo de recetas.

Por otro lado, la regla 5 indica que el aguacate y el limón están comúnmente asociados con la compra de totopos. Esto tiene sentido porque juntos forman el guacamole, una preparación muy popular en la gastronomía mexicana.

Además, el hecho de que estas reglas tengan un lift mayor a 2 indica que se trata de bienes complementarios: es decir, la presencia de uno incrementa significativamente la probabilidad de que se compren los otros, reforzando la idea de que se adquieren de manera conjunta para una misma preparación.

En general, todos estos productos son ingredientes frescos —principalmente frutas, verduras y algunos snacks— que se utilizan en recetas caseras tradicionales. No se trata de alimentos procesados o de marca, sino de insumos básicos que reflejan hábitos de consumo ligados a la cultura culinaria.

3.4 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift.
Regla 1: El soporte del antecedente es 0.32, lo que indica que en el 32% de las compras aparecen juntos la cebolla y el chile jalapeño. De forma similar, el soporte del consecuente también es 0.32, mostrando que el tomate y el cilantro coinciden en ese mismo porcentaje de transacciones. La confianza es de 0.92, lo que sugiere que cuando un cliente incluye cebolla y chile jalapeño, en la gran mayoría de los casos también agrega tomate y cilantro. Además, el lift de 2.90 evidencia que esta relación es mucho más fuerte que una coincidencia aleatoria: la probabilidad de comprar tomate y cilantro aumenta casi tres veces, lo que indica que estos productos funcionan como bienes complementarios.

Regla 3: En este caso, el soporte del antecedente es 0.31, es decir, el 31% de las transacciones contienen cilantro y cebolla. El soporte del consecuente alcanza 0.33, lo que refleja que el chile jalapeño y el tomate aparecen juntos en un 33% de las compras. La confianza es bastante alta (0.95), indicando que casi todos los clientes que compran cilantro y cebolla también adquieren los otros dos ingredientes. Por su parte, el lift de 2.88 confirma que no se trata de una relación al azar, sino de una fuerte complementariedad entre estos productos.

Regla 5: El soporte del antecedente es 0.27, lo que muestra que el 27% de las transacciones incluyen aguacate y limón. El consecuente tiene un soporte de 0.33, señalando que los totopos están presentes en un tercio de las compras. La confianza de 0.93 indica que, cuando se compran aguacate y limón, es muy frecuente que también se incluyan totopos. Finalmente, el lift de 2.82 demuestra que esta asociación es significativa, ya que la compra de totopos es considerablemente más probable en presencia de los otros dos productos, lo que nuevamente refleja una relación de complementariedad.

3.5 **Recomendación de Negocio:** Basado en estas reglas, ¿qué promoción o estrategia de venta específica podrías sugerir para el mercado mexicano?

Las reglas 1 y 3 evidencian que la cebolla, el chile jalapeño, el cilantro y el tomate suelen adquirirse en conjunto, con niveles de confianza superiores al 92%. Esto sugiere una oportunidad clara para diseñar un paquete promocional que incluya los cuatro ingredientes, ofreciendo un descuento entre el 10% y el 15%. Una estrategia así no solo podría aumentar el valor del ticket promedio, sino también hacer más sencilla y rápida la compra para el cliente mexicano.

Por su parte, la regla 5 muestra que cuando se compran aguacate y limón, en el 93% de los casos también se agregan totopos. Este comportamiento respalda la idea de ubicar estos tres productos de manera conjunta en tienda —por ejemplo, en un mismo exhibidor— o destacarlos como combinación en el canal online. Incluso se podría reforzar con mensajes promocionales como: “completa tu guacamole y obtén descuento en totopos”, incentivando la compra complementaria.

3.6 Análisis para Colombia

In [ ]:
# TAREA: Usa la función preparar_cesta para obtener la matriz de transacciones de Colombia. Almacena el resultado en la variable cesta_co.
### TU CÓDIGO AQUÍ ###
cesta_co = preparar_cesta(df_limpio, 'Colombia')
cesta_co.head()

Description,ACEITE DE GIRASOL,ARROZ,AZÚCAR,CAFÉ,FRIJOL CARGAMANTO,HARINA DE MAÍZ,HUEVOS,LECHE,PAN TAJADO,QUESO MUZZARELLA
InvoiceNo,,,,,,,,,,
536000,0,0,0,0,0,1,0,0,0,1
536001,0,0,0,0,0,1,0,0,0,1
536002,0,0,1,1,0,0,0,1,0,0
536003,0,0,0,0,0,0,1,0,1,0
536004,1,1,0,0,1,0,0,0,0,0


In [ ]:
# TAREA: Aplica el algoritmo apriori con un soporte mínimo del 2%.
# Almacena el resultado en la variable frequent_itemsets_co.
# Muestra los 10 itemsets con el soporte más alto.
### TU CÓDIGO AQUÍ ###
frequent_itemsets_co = apriori(cesta_co, min_support=0.02, use_colnames=True)
frequent_itemsets_co.sort_values(by = 'support', ascending=False).head(10)


,support,itemsets
3,0.41,(CAFÉ)
4,0.41,(FRIJOL CARGAMANTO)
0,0.40,(ACEITE DE GIRASOL)
2,0.40,(AZÚCAR)
7,0.39,(LECHE)
1,0.38,(ARROZ)
9,0.35,(QUESO MUZZARELLA)
5,0.34,(HARINA DE MAÍZ)
37,0.32,"(LECHE, CAFÉ)"
31,0.32,"(LECHE, AZÚCAR)"


In [ ]:
# TAREA: Genera las reglas de asociación con un Lift mayor a 2. Almacena el resultado en la variable rules_co.
### TU CÓDIGO AQUÍ ###
rules_co = association_rules(frequent_itemsets_co, metric='lift', min_threshold=2)

In [ ]:
# Ordena las reglas por Lift y Confianza de mayor a menor, muestra solamente las primeras 10 filas y las siguientes columnas:
# 'antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift'
### TU CÓDIGO AQUÍ ###
rules_co.sort_values(by=['lift', 'confidence'], ascending=[False, False]).head(10)[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'confidence', 'lift']]

,antecedents,consequents,antecedent support,consequent support,confidence,lift
42,"(AZÚCAR, FRIJOL CARGAMANTO, CAFÉ)",(LECHE),0.05,0.39,0.96,2.44
43,(LECHE),"(AZÚCAR, FRIJOL CARGAMANTO, CAFÉ)",0.39,0.05,0.11,2.44
12,"(AZÚCAR, CAFÉ)",(LECHE),0.32,0.39,0.95,2.41
13,(LECHE),"(AZÚCAR, CAFÉ)",0.39,0.32,0.76,2.41
5,"(ACEITE DE GIRASOL, FRIJOL CARGAMANTO)",(ARROZ),0.30,0.38,0.91,2.41
8,(ARROZ),"(ACEITE DE GIRASOL, FRIJOL CARGAMANTO)",0.38,0.30,0.73,2.41
60,"(AZÚCAR, PAN TAJADO, CAFÉ)",(LECHE),0.03,0.39,0.94,2.39
65,(LECHE),"(AZÚCAR, PAN TAJADO, CAFÉ)",0.39,0.03,0.07,2.39
50,"(AZÚCAR, CAFÉ, HUEVOS)",(LECHE),0.04,0.39,0.92,2.36
55,(LECHE),"(AZÚCAR, CAFÉ, HUEVOS)",0.39,0.04,0.09,2.36


3.7 Observa las 3 reglas con el Lift más alto para Colombia (1, 3 y 5). **Interprétalas:** ¿Qué patrones de consumo específicos del mercado colombiano revelan estas reglas? ¿Son diferentes a las de México?

Las reglas 1 y 3 reflejan un patrón bastante claro en torno a productos como café, azúcar y leche, que corresponden a una combinación muy común en el consumo diario en Colombia. Aunque la presencia del frijol cargamanto en la regla 1 no es del todo evidente dentro de esa mezcla, podría interpretarse como parte de una compra más amplia, donde los hogares adquieren distintos productos para cubrir varias comidas del día dentro de una misma canasta.

En cuanto a la regla 5, se refuerza aún más esta idea de abastecimiento básico: el aceite de girasol, el frijol y el arroz son alimentos fundamentales en la cocina colombiana, utilizados de forma recurrente en múltiples preparaciones. Esto sugiere que los consumidores no están comprando para una receta específica, sino para mantener surtida la despensa.

En conjunto, estos patrones contrastan con los observados en México. Mientras allá las asociaciones estaban más ligadas a ingredientes frescos para preparaciones concretas como el pico de gallo o el guacamole, en Colombia predominan combinaciones de productos no perecederos y de uso cotidiano, lo que evidencia una lógica de compra más orientada al abastecimiento general del hogar.

3.8 Para cada una de las 3 reglas (1, 3 y 5), interpreta el Soporte para el antecedente y el consecuente, la Confianza y el Lift

Regla 1: El soporte del antecedente es de 0.05, lo que indica que solo el 5% de las transacciones en Colombia incluyen simultáneamente café, frijol y azúcar; es decir, se trata de una combinación poco frecuente pero bastante específica. En contraste, el soporte del consecuente es de 0.39, lo que muestra que la leche está presente en el 39% de las compras y es un producto altamente demandado. La confianza alcanza el 96%, lo que implica que casi siempre que se compran esos tres productos, también se incluye leche. Además, el lift de 2.44 confirma que esta relación no es casual, sino que la presencia de café, frijol y azúcar incrementa significativamente la probabilidad de comprar leche, evidenciando complementariedad.

Regla 3: El antecedente tiene un soporte de 0.32, lo que significa que cerca de un tercio de las transacciones incluyen café y azúcar juntos. El consecuente mantiene un soporte de 0.39, reafirmando el papel central de la leche en la canasta colombiana. La confianza es de 0.95, indicando que la gran mayoría de quienes compran café y azúcar también adquieren leche. El lift de 2.41 muestra que esta relación es más de dos veces superior a lo que se esperaría si los productos fueran independientes, lo que refuerza su carácter complementario.

Regla 5: El soporte del antecedente es de 0.30, lo que indica que el 30% de las transacciones incluyen aceite de girasol y frijol cargamanto, reflejando su importancia como productos básicos. El soporte del consecuente es de 0.38, evidenciando que el arroz también tiene una fuerte presencia en las compras. La confianza de 0.91 señala que la mayoría de los clientes que compran aceite y frijol también agregan arroz. Por último, el lift de 2.41 confirma que esta asociación tiene un valor predictivo relevante y no responde al azar, mostrando nuevamente una relación de complementariedad dentro de la canasta básica.

3.9 **Recomendación de Negocio:** ¿Qué campaña de marketing (diferente a la de México) podrías diseñar para los clientes colombianos?
Las reglas 1 y 3 dejan ver que café, azúcar y leche funcionan casi como un combo natural dentro del consumo colombiano, con niveles de confianza entre el 95% y el 96%. Esto abre una oportunidad clara para ofrecerlos como un paquete conjunto —por ejemplo, bajo el concepto de “kit del desayuno perfecto”— acompañado de un descuento por compra combinada. Dado que productos como el café y la leche tienen soportes altos (32% y 39%), la estrategia tendría un alcance amplio y podría impactar a una gran parte de los clientes.

Por su parte, la regla 5 muestra un patrón muy consistente entre arroz, frijol cargamanto y aceite de girasol. Estos productos podrían agruparse en tienda física o en el canal digital bajo una categoría como “todo para tu almuerzo”, incluso complementando con recomendaciones adicionales como proteínas o verduras. Con esto, no solo se incentiva la compra cruzada, sino que también se posiciona la plataforma como una opción práctica para surtir la despensa semanal.

En comparación con México, donde las promociones funcionan mejor cuando están ligadas a recetas específicas y tradicionales, en Colombia resulta más efectivo enfocarse en la lógica de abastecimiento del hogar. Es decir, resaltar el ahorro, la practicidad y la posibilidad de completar la canasta básica en una sola compra.